In [ ]:
import time
import torch
import numpy as np
from typing import Optional, Tuple
from dataclasses import dataclass
import gc
import os

@dataclass
class GradConfig:
    num_steps: int = 1000
    num_samples: int = 100
    temperature: float = 0.1
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: Optional[int] = None


class GradientDescent:
    def __init__(self, T, J, h, config: GradConfig):
        self.config = config
        self.device = torch.device(config.device)
        
        self.T = torch.tensor(T, device=self.device, dtype=torch.float32)
        self.J = torch.tensor(J, device=self.device, dtype=torch.float32)
        self.h = torch.tensor(h, device=self.device, dtype=torch.float32)
        self.N = self.J.shape[0]
        
    
    def _energy(self, s):
        
        E = -torch.einsum('ijk,bi,bj,bk->b', self.T, s, s, s)
        E -= torch.einsum('ij,bi,bj->b', self.J, s, s)
        E -= torch.einsum('i,bi->b', self.h, s)
        return E
    
    def _gradient(self, s):
        
        grad = torch.zeros_like(s)
        
        
        for i in range(self.N):
            grad[:, i] -= torch.einsum('jk,bj,bk->b', self.T[i], s, s)
        
        grad -= torch.mm(s, self.J.T)
        
        grad -= self.h
        
        return grad
    
    def solve(self):
        
        s = torch.randint(0, 2, (self.config.num_samples, self.N), 
                         device=self.device) * 2 - 1
        s = s.float()
        
        best_energy = float('inf')
        best_s = None
        data = []
        
        for step in range(self.config.num_steps):
            
            grad = self._gradient(s)
            
            target = torch.sign(-grad)

            s = target
            if step % 10 == 0:
                spins = torch.sign(s)
                data.append(s.cpu())
                energy = self._energy(s)
                min_energy = energy.min().item()
                if min_energy < best_energy:
                    best_energy = min_energy
                    best_idx = energy.argmin().item()
                    best_s = s[best_idx].cpu().numpy()
        
        return best_s, data




In [ ]:
def compute_energy_sparse(x, H1, H2, H3):

    energy = 0.0
    n = len(x)
    
    # Линейные
    energy += np.dot(x, H1)
    
    # Квадратичные (только верхний треугольник)
    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]
    
    # Кубические (только i < j < k)
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy

In [ ]:
for i in range(20):
    torch.cuda.empty_cache()
    gc.collect()
    H1=np.zeros(200) 
    H2=np.zeros((200, 200)) 
    H3=np.load(f"J_T_h_1/T_{i}.npy")
       
    config = GradConfig(num_steps=500, num_samples=5000, temperature=0.05, device="cuda")
    solver = GradientDescent(-H3, -H2, -H1, config)
    spins, data = solver.solve()
    print(i, compute_energy_sparse(spins, H1, H2, H3))
    np.save(f"solution_data_1/greedy_{i}.npy", data)